# Investment AI
## Notebook 01 — Unity Catalog Foundation

In Notebook 00, we validated that the Databricks environment is healthy.

Now we create the governed Unity Catalog foundation for the project.

For this POC, we will use a dedicated catalog:

```text
investment_ai
├── research
├── portfolio
└── agent
```

This gives the project a clean top-level boundary instead of placing everything under the default `workspace` catalog.

Each schema has a clear responsibility:

- `investment_ai.research` → research documents, parsed content, chunks, and AI Search source tables
- `investment_ai.portfolio` → structured portfolio and market data
- `investment_ai.agent` → agent-facing functions, semantic assets, prompts, and application resources

We will also create Unity Catalog Volumes for the source files that will be landed in the next notebook.


## Why use a dedicated catalog?

A dedicated catalog gives the POC a cleaner project boundary.

Instead of:

```text
workspace.investment_ai_research
workspace.investment_ai_portfolio
workspace.investment_ai_agent
```

we can use:

```text
investment_ai.research
investment_ai.portfolio
investment_ai.agent
```

This is easier to understand, easier to govern, and closer to how we would organize a larger production project.


In [0]:
CATALOG = "investment_ai"

RESEARCH_SCHEMA = "research"
PORTFOLIO_SCHEMA = "portfolio"
AGENT_SCHEMA = "agent"

print("Catalog          :", CATALOG)
print("Research schema  :", RESEARCH_SCHEMA)
print("Portfolio schema :", PORTFOLIO_SCHEMA)
print("Agent schema     :", AGENT_SCHEMA)


## 1. Create the dedicated catalog

This is the top-level Unity Catalog container for the entire Investment AI project.

The command is idempotent, so it is safe to rerun.

> Note: your Databricks identity must have permission to create catalogs.


In [0]:
spark.sql(
    f"CREATE CATALOG IF NOT EXISTS {CATALOG}"
)

print(f"✅ Catalog created or already exists: {CATALOG}")


## 2. Validate the catalog

Before creating schemas, confirm that the new catalog is visible.


In [0]:
catalogs = [row["catalog"] for row in spark.sql("SHOW CATALOGS").collect()]

assert CATALOG in catalogs, (
    f"Catalog '{CATALOG}' is not available. "
    f"Visible catalogs: {catalogs}"
)

print(f"✅ Catalog '{CATALOG}' is available.")


## 3. Create the project schemas

We separate research, portfolio, and agent assets so they do not all live in the same namespace.

This also gives us cleaner governance boundaries later.


In [0]:
schemas = [
    RESEARCH_SCHEMA,
    PORTFOLIO_SCHEMA,
    AGENT_SCHEMA,
]

for schema_name in schemas:
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}"
    )
    print(f"✅ {CATALOG}.{schema_name}")


## 4. Create Unity Catalog Volumes

We will use Volumes as the governed file-storage layer for the POC.

The research schema will contain:
- raw research documents

The portfolio schema will contain:
- structured CSV/JSON seed files
- semi-structured Excel files


In [0]:
volumes = [
    (RESEARCH_SCHEMA, "raw_documents"),
    (PORTFOLIO_SCHEMA, "seed_data"),
    (PORTFOLIO_SCHEMA, "semi_structured"),
]

for schema_name, volume_name in volumes:
    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS "
        f"{CATALOG}.{schema_name}.{volume_name}"
    )
    print(
        f"✅ /Volumes/{CATALOG}/{schema_name}/{volume_name}"
    )


## 5. Verify the schemas


In [0]:
all_schemas = spark.sql(
    f"SHOW SCHEMAS IN {CATALOG}"
)

display(all_schemas)


## 6. Verify the Volumes


In [0]:
research_volumes = spark.sql(
    f"SHOW VOLUMES IN {CATALOG}.{RESEARCH_SCHEMA}"
)

portfolio_volumes = spark.sql(
    f"SHOW VOLUMES IN {CATALOG}.{PORTFOLIO_SCHEMA}"
)

print("Research volumes")
display(research_volumes)

print("Portfolio volumes")
display(portfolio_volumes)


## 7. Build the source paths we will use later

These paths will be referenced in the source-landing notebook.


In [0]:
RAW_DOCUMENTS_PATH = (
    f"/Volumes/{CATALOG}/{RESEARCH_SCHEMA}/raw_documents"
)

SEED_DATA_PATH = (
    f"/Volumes/{CATALOG}/{PORTFOLIO_SCHEMA}/seed_data"
)

SEMI_STRUCTURED_PATH = (
    f"/Volumes/{CATALOG}/{PORTFOLIO_SCHEMA}/semi_structured"
)

print("Research documents :", RAW_DOCUMENTS_PATH)
print("Portfolio seed data :", SEED_DATA_PATH)
print("Semi-structured     :", SEMI_STRUCTURED_PATH)


## 8. Final validation

The expected project foundation is:

```text
investment_ai
│
├── research
│   └── raw_documents
│
├── portfolio
│   ├── seed_data
│   └── semi_structured
│
└── agent
```

At this stage, we have created only the governed containers.

We have **not loaded any data yet**.

That is intentional.

The next notebook will take the source files from the Git repository and land them into these Unity Catalog Volumes.


In [0]:
expected_schemas = {
    RESEARCH_SCHEMA,
    PORTFOLIO_SCHEMA,
    AGENT_SCHEMA,
}

visible_schemas = {
    row["databaseName"]
    for row in spark.sql(
        f"SHOW SCHEMAS IN {CATALOG}"
    ).collect()
}

missing = expected_schemas - visible_schemas

assert not missing, f"Missing schemas: {missing}"

print("✅ Unity Catalog foundation is ready.")
print("✅ Dedicated catalog created.")
print("✅ Research, portfolio, and agent schemas created.")
print("✅ Source Volumes created.")
print("\nNext: 02_land_source_files.ipynb")
